# LKA 完整五折训练与评估
按顺序运行；默认CAN双路径。图像模型不提供本地预训练权重时采用随机初始化。

单元格1：设置项目位置与Python

In [ ]:
from pathlib import Path
from datetime import datetime
import os
import subprocess
import pandas as pd
from IPython.display import display

PROJECT = Path.cwd().resolve()
if not (PROJECT / 'scripts/train_and_evaluate.py').exists():
    PROJECT = PROJECT / 'lka_project'
import sys
PYTHON = Path(sys.executable)
os.chdir(PROJECT)
assert (PROJECT / 'scripts/train_and_evaluate.py').exists()
assert PYTHON.exists()
print('项目目录：', PROJECT)
print('训练使用的Python：', PYTHON)



单元格2：选择模型并设置完整五折训练参数

In [ ]:
MODEL = 'dual'  # stats、tcn、dual、image、fusion、gated
EPOCHS = 40
BATCH_SIZE = 32  # 图像或门控模型可先用16
LEARNING_RATE = 0.001  # 图像或门控模型可从0.0001开始验证
WEIGHTS = None  # 使用预训练ResNet时，填写本地权重文件的绝对路径
RUN_DIR = PROJECT / 'runs' / f'{MODEL}_cv_{datetime.now():%Y%m%d_%H%M%S_%f}'
print('本次输出目录：', RUN_DIR)



单元格3：启动训练，实时显示日志，结束后自动评估

In [ ]:
command = [str(PYTHON), '-u', 'scripts/train_and_evaluate.py',
           '--model', MODEL, '--fold', 'all', '--epochs', str(EPOCHS),
           '--patience', '7', '--batch-size', str(BATCH_SIZE),
           '--lr', str(LEARNING_RATE), '--output', str(RUN_DIR)]
if WEIGHTS:
    command += ['--weights', str(WEIGHTS)]
process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise RuntimeError('训练未正常完成，请检查上方错误。')
except KeyboardInterrupt:
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    raise
print('\n五折训练与评估已完成：', RUN_DIR)



单元格4：查看五折指标与合并预测的指标

In [ ]:
metrics = ['accuracy', 'precision', 'recall', 'f1', 'f2',
           'roc_auc', 'average_precision', 'pr_auc_trapezoid']
print('各折验证指标：')
display(pd.read_csv(RUN_DIR / 'cv_metrics.csv')[['fold'] + metrics])
print('五折均值与标准差：')
display(pd.read_csv(RUN_DIR / 'cv_summary.csv', index_col=0))
print('五折验证预测合并后的整体指标：')
display(pd.read_csv(RUN_DIR / 'pooled_validation/metrics.csv')[metrics])
print('合并混淆矩阵（行=真实，列=预测）：')
display(pd.read_csv(RUN_DIR / 'pooled_validation/confusion_matrix.csv', index_col=0))



可选单元格5：画出五折训练曲线

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for fold in range(5):
    history = pd.read_csv(RUN_DIR / f'fold{fold}/training_history.csv')
    axes[0].plot(history['epoch'], history['train_loss_batch_weighted'], label=f'Fold {fold+1}')
    axes[1].plot(history['epoch'], history['val_ap'], label=f'Fold {fold+1}')
axes[0].set(xlabel='Epoch', ylabel='Training loss')
axes[1].set(xlabel='Epoch', ylabel='Validation AP')
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(RUN_DIR / 'five_fold_training_curves.png', dpi=200)
plt.show()
